In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os


feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
# feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURE_ENGINEERING.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *

### Load Model

In [7]:
model = joblib.load('Models/xgbModel.pkl')
features = joblib.load('Models/top_features.pkl')

### Load Data

In [8]:
pd.set_option('display.max_columns', None)
eplison = 0.000001

s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
s25['EXPECTED_USAGE_MIN'] = s25['USG_PCT_ROLLING_AVG_5'] * (s25['MIN_ROLLING_AVG_5'] + eplison)

s24 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')
s24['EXPECTED_USAGE_MIN'] = s24['USG_PCT_ROLLING_AVG_5'] * (s24['MIN_ROLLING_AVG_5'] + eplison)
df = pd.concat([s25, s24]).sort_values(by='GAME_DATE')


date = '2025-03-11'
df = df[df['GAME_DATE'] < date]

dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')
dfsData = dfsData[(dfsData['BOOKMAKER'] == 'prizepicks') & (dfsData['GAME_DATE'] == date) & (dfsData['CATEGORY'] == 'player_points')]
df.tail()

C:\Users\alexg\AppData\Local\Temp\ipykernel_18720\4012596075.py:15: DtypeWarning: Columns (11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')


Unnamed: 0  Unnamed: 0.2         PLAYER_NAME  PLAYER_ID      MATCHUP  \
12352       12352       20572.0          RJ Barrett    1629628  TOR vs. WAS   
12337       12337       20595.0  Isaiah Hartenstein    1628392  OKC vs. DEN   
12338       12338       20597.0          Isaiah Joe    1630198  OKC vs. DEN   
12339       12339       20598.0      Jalen Williams    1631114  OKC vs. DEN   
12336       12336       20602.0        Dillon Jones    1641794  OKC vs. DEN   

      TEAM_ABBREVIATION     TEAM_ID OPP_ABBREVIATION  HOME_GAME   GAME_ID  \
12352               TOR  1610612761              WAS          1  22400932   
12337               OKC  1610612760              DEN          1  22400936   
12338               OKC  1610612760              DEN          1  22400936   
12339               OKC  1610612760              DEN          1  22400936   
12336               OKC  1610612760              DEN          1  22400936   

        GAME_DATE WL  PTS  AST  REB  FGM  FGA  FG_PCT  FG3M  FG3A  FG3_PCT  \
12352  2025-03-10  W   14    8   10    6   18   0.333     1     6    0.167   
12337  2025-03-10  L   20    2    7    8   16   0.500     0     1    0.000   
12338  2025-03-10  L    3    3    2    1    5   0.200     1     3    0.333   
12339  2025-03-10  L   12    6    2    5   10   0.500     1     3    0.333   
12336  2025-03-10  L    0    0    0    0    0     NaN     0     0      NaN   

       FTM  FTA  FT_PCT  OREB  DREB  STL  BLK  TOV  PLUS_MINUS  FANTASY_PTS  \
12352    1    4    0.25     4     6    0    0    2          14         36.0   
12337    4    4    1.00     2     5    0    0    0           1         31.4   
12338    0    0     NaN     2     0    0    0    1           4          8.9   
12339    1    2    0.50     0     2    0    0    1           8         22.4   
12336    0    0     NaN     0     0    0    0    0           3          0.0   

       POINT_PER_SHOT       EFG START_POSITION  COMMENT  E_OFF_RATING  \
12352           0.709  0.361111              F      NaN         130.1   
12337           1.126  0.500000              C      NaN         135.4   
12338           0.600  0.300000            NaN      NaN         126.6   
12339           1.103  0.550000              F      NaN         137.2   
12336           0.000       NaN            NaN      NaN         166.7   

       E_DEF_RATING  NET_RATING  OREB_PCT  DREB_PCT  REB_PCT  AST_PCT  \
12352         102.9        28.2     0.089     0.162    0.122    0.333   
12337         141.2         7.2     0.077     0.250    0.152    0.105   
12338         130.0         8.2     0.100     0.000    0.065    0.188   
12339         123.8        22.5     0.000     0.133    0.077    0.500   
12336         100.0       100.0     0.000     0.000    0.000    0.000   

       EFG_PCT  AST_TOV  USG_PCT  TS_PCT  E_PACE    PACE    PIE  POSS  \
12352    0.361      4.0    0.272   0.354  104.78  102.42  0.101    57   
12337    0.500      0.0    0.274   0.563  100.80   99.12  0.104    52   
12338    0.300      3.0    0.146   0.300   97.07   99.26  0.000    34   
12339    0.550      6.0    0.364   0.551  108.33  104.50  0.174    30   
12336    0.000      0.0    0.000   0.000   67.92   81.51  0.000     3   

       PACE_PER40  E_USG_PCT    MIN   SPD  DIST  ORBC  DRBC  RBC  TCHS  SAST  \
12352       85.35      0.274  27.18  4.47  2.14     7     7   14    63     0   
12337       82.60      0.272  25.67  4.42  2.02     6     7   13    49     0   
12338       82.72      0.147  16.20  4.36  1.27     2     0    2    31     0   
12339       87.08      0.352  13.55  4.41  1.06     0     3    3    31     0   
12336       67.92      0.000   1.77  3.72  0.12     0     0    0     4     0   

       FTAST  PASS  CFGM  CFGA  CFG_PCT  UFGM  UFGA  UFG_PCT  DFGM  DFGA  \
12352      0    41     3     8    0.375     3    10    0.300     1     1   
12337      0    32     3     5    0.600     5    11    0.455     1     2   
12338      0    26     0     1    0.000     1     4    0.250     0     0   
12339      0    17     

In [9]:
backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
backtestData = backtestData[(backtestData['ODDS'] <= 200) & (backtestData['ODDS'] >= -200)]
singleBookies = backtestData[(backtestData['CATEGORY'] == 'points') & (backtestData['GAME_DATE'] == date)]
singleBookies

,NAME,CATEGORY,SIDE,BOOKMAKER,LINE,ODDS,fair_line,fair_odds,GAME_DATE
124606,Draymond Green,points,under,fanduel,10.5,-122,10.5,102,2025-03-11
124607,Draymond Green,points,under,draftkings,10.5,-110,10.5,102,2025-03-11
124608,Draymond Green,points,under,espnbet,10.5,-120,10.5,102,2025-03-11
124610,Draymond Green,points,under,betrivers,10.5,-113,10.5,102,2025-03-11
124616,Draymond Green,points,over,fanduel,10.5,-104,10.5,-102,2025-03-11
...,...,...,...,...,...,...,...,...,...
127136,Karlo Matković,points,over,espnbet,6.5,-105,6.5,106,2025-03-11
127137,Karlo Matković,points,over,draftkings,6.5,-105,6.5,106,2025-03-11
127144,Karlo Matković,points,under,betmgm,6.5,-125,6.5,-106,2025-03-11
127145,Karlo Matković,points,under,espnbet,6.5,-125,6.5,-106,2025-03-11


### Top EVs for single bets

In [10]:
results = single_bet(
    data=df,
    bookmakers=singleBookies,
    model=model,
    features=features,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
560,Taurean Prince,betmgm,points,17.5,100,under,6.283136,1,0.018,0.982,0.500,96.48,0.96,0.48,0.24,"(0.6, 16.8)"
655,Zion Williamson,espnbet,points,22.5,-105,over,27.931557,1,0.971,0.029,0.512,89.62,0.94,0.47,0.24,"(22.3, 33.7)"
658,Zion Williamson,betrivers,points,22.5,-112,over,27.931557,1,0.967,0.033,0.528,83.02,0.93,0.46,0.23,"(22.2, 33.6)"
741,Amir Coffey,draftkings,points,7.5,-110,over,11.032525,0,0.948,0.052,0.524,81.06,0.89,0.45,0.22,"(6.8, 15.3)"
739,Amir Coffey,espnbet,points,7.5,-110,over,11.032525,0,0.947,0.053,0.524,80.71,0.89,0.44,0.22,"(6.8, 15.3)"
742,Amir Coffey,fanduel,points,7.5,-110,over,11.032525,0,0.946,0.054,0.524,80.58,0.89,0.44,0.22,"(6.8, 15.3)"
654,Zion Williamson,draftkings,points,22.5,-120,over,27.931557,1,0.971,0.029,0.545,78.09,0.94,0.47,0.23,"(22.4, 33.6)"
657,Zion Williamson,betmgm,points,22.5,-120,over,27.931557,1,0.971,0.029,0.545,77.96,0.94,0.47,0.23,"(22.3, 33.7)"
374,Javonte Green,betmgm,points,0.5,-120,over,2.307379,0,0.960,0.040,0.545,75.96,0.91,0.46,0.23,"(0.3, 18.1)"
740,Amir Coffey,betmgm,points,7.5,-120,over,11.032525,0,0.951,0.049,0.545,74.37,0.89,0.45,0.22,"(6.9, 15.3)"


### Top EVs for 2 leg bets

In [11]:
results = prizepickspairsEV(
    data=df,
    bookmakers=dfsData,
    model=model,
    features=features,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,SIDE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,SIDE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.099,0.901,"(3.4, 28.6)",UNDER/UNDER,1,0.8872,1.661,0.831
1,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.899,0.101,"(2.4, 15.8)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",OVER/UNDER,0,0.8843,1.653,0.826
2,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",Obi Toppin,player_points,prizepicks,-137,7.5,under,12.55,OVER,0.867,0.133,"(3.8, 21.8)",UNDER/OVER,1,0.8536,1.561,0.780
3,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",Max Christie,player_points,prizepicks,-137,16.5,under,13.39,UNDER,0.137,0.863,"(7.8, 19.0)",UNDER/UNDER,0,0.8489,1.547,0.773
4,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,7.5,under,4.49,UNDER,0.143,0.857,"(0.5, 9.9)",UNDER/UNDER,0,0.8437,1.531,0.766
5,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",Patrick Williams,player_points,prizepicks,-137,8.5,over,4.05,UNDER,0.149,0.851,"(0.3, 12.0)",UNDER/UNDER,0,0.8375,1.512,0.756
6,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",Julian Champagnie,player_points,prizepicks,-137,7.5,over,12.30,OVER,0.848,0.152,"(3.4, 21.7)",UNDER/OVER,1,0.8345,1.504,0.752
7,Ja Morant,player_points,prizepicks,-137,27.0,over,21.13,UNDER,0.158,0.842,"(9.8, 32.5)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",UNDER/UNDER,1,0.8286,1.486,0.743
8,Harrison Barnes,player_points,prizepicks,-137,11.0,over,18.98,OVER,0.839,0.161,"(4.0, 35.8)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",OVER/UNDER,1,0.8258,1.477,0.739
9,Ausar Thompson,player_points,prizepicks,-137,11.5,over,6.66,UNDER,0.167,0.833,"(0.7, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.016,0.984,"(1.6, 20.4)",UNDER/UNDER,1,0.8194,1.458,0.729


In [5]:
threeLeg = prizepicks3LegEV(
    data=df,
    bookmakers=dfsData,
    model=model,
    features=features,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,SIDE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,SIDE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,SIDE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
105502,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",OVER/UNDER/UNDER,0,0.8030,3.818,0.764
351546,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Max Christie,player_points,prizepicks,-137,16.5,under,13.39,UNDER,0.133,0.867,"(7.9, 18.9)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",UNDER/UNDER/UNDER,0,0.7726,3.636,0.727
105487,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Max Christie,player_points,prizepicks,-137,16.5,under,13.39,UNDER,0.133,0.867,"(7.9, 18.9)",OVER/UNDER/UNDER,0,0.7697,3.618,0.724
352051,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Obi Toppin,player_points,prizepicks,-137,7.5,under,12.55,OVER,0.862,0.138,"(3.7, 21.7)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",UNDER/OVER/UNDER,1,0.7686,3.612,0.722
105500,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Obi Toppin,player_points,prizepicks,-137,7.5,under,12.55,OVER,0.862,0.138,"(3.7, 21.7)",OVER/UNDER/OVER,0,0.7657,3.594,0.719
352117,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",Patrick Williams,player_points,prizepicks,-137,8.5,over,4.05,UNDER,0.147,0.853,"(0.3, 11.8)",UNDER/UNDER/UNDER,0,0.7608,3.565,0.713
350812,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,7.5,under,4.49,UNDER,0.147,0.853,"(0.5, 9.9)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",UNDER/UNDER/UNDER,0,0.7602,3.561,0.712
105505,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Patrick Williams,player_points,prizepicks,-137,8.5,over,4.05,UNDER,0.147,0.853,"(0.3, 11.8)",OVER/UNDER/UNDER,0,0.7579,3.547,0.709
105473,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,7.5,under,4.49,UNDER,0.147,0.853,"(0.5, 9.9)",OVER/UNDER/UNDER,0,0.7574,3.544,0.709
350566,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Julian Champagnie,player_points,prizepicks,-137,7.5,over,12.30,OVER,0.844,0.156,"(3.4, 21.6)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",UNDER/OVER/UNDER,1,0.7526,3.515,0.703
